## Final Model Comparison

This notebook compares the logistic regression models in a more structured way, using train-test split, cross-validation, gap analysis, hyperparameter tuning, threshold optimization, and final test evaluation.

The objective is to select a final model based not only on test performance, but also on generalization stability, interpretability, and business relevance.

In [55]:
import importlib

import pandas as pd
import numpy as np

import attrition_analysis.data_selection as data_selection
import attrition_analysis.models_utils as models_utils

importlib.reload(data_selection)
importlib.reload(models_utils)

from attrition_analysis.models_utils import (
    split_train_test_df,
    estimators_dict,
    param_distributions_dict,
    categorical_models_dict_c,
    mixed_models_dict_c,
    run_logistic_cross_validation,
    run_logistic_gap_analysis,
    run_logistic_model_comparison,
    tune_logistic_hyperparameters_top_combinations,
    evaluate_thresholds_optimized_logistic_models_cv
)


df = pd.read_csv("../../data/clean/Employee-Attrition_Clean.csv")

target = "AttritionFlag"

df.head()

,Age,Attrition,BusinessTravel,DailyRate,Department,DistanceFromHome,EducationField,EmployeeCount,EmployeeNumber,Gender,...,IncomeGroup,EducationLevel,StockOption,JobLevelGroup,SatisfactionLevel,PerformanceRatingLevel,EnvironmentSatisfactionLevel,JobInvolvementLevel,RelationshipSatisfactionLevel,WorkLifeBalanceLevel
0,41,Yes,Travel_Rarely,1102,Sales,1,Life Sciences,1,1,Female,...,Medium,College,No Options,Junior Level,Very High,Excellent,Medium,High,Low,Bad
1,49,No,Travel_Frequently,279,Research & Development,8,Life Sciences,1,2,Male,...,Medium,Below College,Low,Junior Level,Medium,Outstanding,High,Medium,Very High,Better
2,37,Yes,Travel_Rarely,1373,Research & Development,2,Other,1,4,Male,...,Low,College,No Options,Entry Level,High,Excellent,Very High,Medium,Medium,Better
3,33,No,Travel_Frequently,1392,Research & Development,3,Life Sciences,1,5,Female,...,Low,Master,No Options,Entry Level,High,Excellent,Very High,High,High,Better
4,27,No,Travel_Rarely,591,Research & Development,2,Medical,1,7,Male,...,Medium,Below College,Low,Entry Level,Medium,Excellent,Low,High,Very High,Better


In [56]:
all_logistic_models_dict_c = {
    **categorical_models_dict_c,
    **mixed_models_dict_c
}

len(all_logistic_models_dict_c)

15

### Train-test split and class distribution

The dataset was split into training and test sets using a 70/30 split. The class distribution was preserved in both sets, with approximately 83.9% of employees belonging to the no-attrition class and 16.1% belonging to the attrition class.

This confirms that the split is representative of the original class imbalance. Since attrition is the minority class, accuracy alone is not sufficient to evaluate model performance. Metrics such as recall, precision, F1-score, AUC, and the confusion matrix are more informative for this problem.

In [57]:
df_train, df_test = split_train_test_df(
    df=df,
    target=target,
    test_size=0.30,
    random_state=42
)

print("Train shape:", df_train.shape)
print("Test shape:", df_test.shape)

print("\nTrain target distribution:")
display(df_train[target].value_counts(normalize=True).round(3))

print("\nTest target distribution:")
display(df_test[target].value_counts(normalize=True).round(3))

Train shape: (1029, 43)
Test shape: (441, 43)

Train target distribution:


AttritionFlag
0    0.839
1    0.161
Name: proportion, dtype: float64


Test target distribution:


AttritionFlag
0    0.839
1    0.161
Name: proportion, dtype: float64

### Cross-validation results

Cross-validation was used to evaluate model stability across different folds of the training data.

The integrated multidimensional model achieved the highest cross-validated F1-score, confirming its strength as a benchmark model. However, this model includes many predictors and is affected by higher complexity and multicollinearity.

Among the more interpretable candidates, the hierarchical and benefits model with balanced logistic regression showed strong performance, with a good recall and stable cross-validation behavior. This supports its use as a final explanatory model.

In [58]:
cv_comparison = run_logistic_cross_validation(
    df=df_train,
    models_dict=all_logistic_models_dict_c,
    estimators_dict=estimators_dict,
    target=target,
    n_splits=10,
    random_state=42,
    scale_numeric=True
)

cv_comparison_sorted = (
    cv_comparison
    .sort_values(
        by=["F1_Mean", "Recall_Mean", "AUC_Mean"],
        ascending=False
    )
    .reset_index(drop=True)
)

cv_comparison_sorted

,Variable_Set,Model,Accuracy_Mean,Accuracy_Std,Precision_Mean,Precision_Std,Recall_Mean,Recall_Std,F1_Mean,F1_Std,AUC_Mean,AUC_Std,N_Numeric_Variables,N_Categorical_Variables,N_Features_After_Dummies
0,Modelo 8 — Integrado Multidimensional,Logistic Regression Balanced,0.783,0.038,0.410,0.061,0.758,0.107,0.531,0.071,0.847,0.069,7,11,43
1,Modelo 8 — Integrado Multidimensional,Logistic Regression,0.880,0.022,0.738,0.138,0.422,0.112,0.526,0.105,0.856,0.067,7,11,43
2,Modelo 2 — Nível Hierárquico e Benefícios,Logistic Regression Balanced,0.748,0.027,0.360,0.032,0.716,0.102,0.478,0.044,0.813,0.068,3,8,25
3,Modelo 4 — Trajetória Organizacional,Logistic Regression Balanced,0.744,0.048,0.358,0.064,0.721,0.140,0.477,0.083,0.796,0.074,0,8,22
4,Modelo 5 — Antiguidade Organizacional,Logistic Regression Balanced,0.726,0.044,0.342,0.046,0.733,0.102,0.464,0.053,0.783,0.071,5,6,20
5,Modelo 6 — Perfil Pessoal,Logistic Regression Balanced,0.741,0.029,0.348,0.049,0.704,0.152,0.464,0.075,0.790,0.073,0,9,24
6,Modelo 4 — Experiência Profissional,Logistic Regression Balanced,0.729,0.042,0.338,0.065,0.721,0.163,0.459,0.092,0.791,0.075,4,6,19
7,Modelo 5 — Estabilidade e Benefícios,Logistic Regression Balanced,0.734,0.054,0.347,0.057,0.692,0.100,0.459,0.059,0.795,0.066,0,9,24
8,Modelo 6 — Perfil Pessoal e Condições de Trabalho,Logistic Regression Balanced,0.735,0.028,0.341,0.043,0.703,0.132,0.458,0.066,0.791,0.072,3,8,24
9,Modelo 1 — Função Profissional,Logistic Regression Balanced,0.726,0.058,0.339,0.077,0.697,0.109,0.455,0.089,0.789,0.056,0,8,26


### Gap analysis

The gap analysis compares training and cross-validation performance in order to identify potential overfitting or underfitting.

The selected hierarchical and benefits model with balanced logistic regression showed stable generalization, with a relatively small gap between training and cross-validation F1-score and recall.

This indicates that the model is not simply fitting the training data, but is able to generalize reasonably well across validation folds.

In [59]:
gap_analysis = run_logistic_gap_analysis(
    df=df_train,
    models_dict=all_logistic_models_dict_c,
    estimators_dict=estimators_dict,
    target=target,
    n_splits=10,
    random_state=42,
    scale_numeric=True
)

gap_analysis_sorted = (
    gap_analysis
    .sort_values(
        by=["CV_F1_Mean", "CV_Recall_Mean", "CV_AUC_Mean"],
        ascending=False
    )
    .reset_index(drop=True)
)

gap_analysis_sorted

,Variable_Set,Model,Train_F1_Mean,CV_F1_Mean,F1_Gap,Train_Recall_Mean,CV_Recall_Mean,Recall_Gap,Train_AUC_Mean,CV_AUC_Mean,AUC_Gap,Gap_Diagnosis,N_Numeric_Variables,N_Categorical_Variables,N_Features_After_Dummies
0,Modelo 8 — Integrado Multidimensional,Logistic Regression Balanced,0.571,0.531,0.041,0.823,0.758,0.065,0.893,0.847,0.046,Stable generalization,7,11,43
1,Modelo 8 — Integrado Multidimensional,Logistic Regression,0.606,0.526,0.080,0.483,0.422,0.062,0.890,0.856,0.033,Moderate gap,7,11,43
2,Modelo 2 — Nível Hierárquico e Benefícios,Logistic Regression Balanced,0.515,0.478,0.037,0.780,0.716,0.064,0.840,0.813,0.027,Stable generalization,3,8,25
3,Modelo 4 — Trajetória Organizacional,Logistic Regression Balanced,0.507,0.477,0.030,0.762,0.721,0.041,0.827,0.796,0.031,Stable generalization,0,8,22
4,Modelo 5 — Antiguidade Organizacional,Logistic Regression Balanced,0.480,0.464,0.016,0.760,0.733,0.027,0.811,0.783,0.028,Stable generalization,5,6,20
5,Modelo 6 — Perfil Pessoal,Logistic Regression Balanced,0.504,0.464,0.040,0.757,0.704,0.053,0.825,0.790,0.035,Stable generalization,0,9,24
6,Modelo 4 — Experiência Profissional,Logistic Regression Balanced,0.492,0.459,0.034,0.765,0.721,0.044,0.814,0.791,0.023,Stable generalization,4,6,19
7,Modelo 5 — Estabilidade e Benefícios,Logistic Regression Balanced,0.508,0.459,0.049,0.762,0.692,0.070,0.832,0.795,0.037,Stable generalization,0,9,24
8,Modelo 6 — Perfil Pessoal e Condições de Trabalho,Logistic Regression Balanced,0.500,0.458,0.041,0.756,0.703,0.053,0.828,0.791,0.037,Stable generalization,3,8,24
9,Modelo 1 — Função Profissional,Logistic Regression Balanced,0.482,0.455,0.027,0.740,0.697,0.043,0.831,0.789,0.042,Stable generalization,0,8,26


In [60]:
general_results, threshold_results, confusion_results, trained_models, interpretation_results = run_logistic_model_comparison(
    df=df_train,
    models_dict=all_logistic_models_dict_c,
    estimators_dict=estimators_dict,
    target=target,
    thresholds=np.arange(0.20, 0.651, 0.025),
    test_size=0.30,
    random_state=42,
    scale_numeric=True
)

general_results_sorted = (
    general_results
    .sort_values(
        by=["F1-score", "Recall", "AUC"],
        ascending=False
    )
    .reset_index(drop=True)
)

general_results_sorted

,Variable_Set,Model,Threshold,Accuracy,Precision,Recall,F1-score,AUC,N_Numeric_Variables,N_Categorical_Variables,N_Features_After_Dummies
0,Modelo 8 — Integrado Multidimensional,Logistic Regression Balanced,0.5,0.809,0.446,0.74,0.556,0.823,7,11,43
1,Modelo 2 — Nível Hierárquico e Benefícios,Logistic Regression Balanced,0.5,0.764,0.384,0.76,0.510,0.808,3,8,25
2,Modelo 6 — Perfil Pessoal e Condições de Trabalho,Logistic Regression Balanced,0.5,0.751,0.361,0.70,0.476,0.786,3,8,24
3,Modelo 4 — Trajetória Organizacional,Logistic Regression Balanced,0.5,0.741,0.347,0.68,0.459,0.782,0,8,22
4,Modelo 6 — Perfil Pessoal,Logistic Regression Balanced,0.5,0.741,0.347,0.68,0.459,0.774,0,9,24
5,Modelo 1 — Função Profissional Misto,Logistic Regression Balanced,0.5,0.751,0.348,0.62,0.446,0.779,3,7,26
6,Modelo 1 — Função Profissional,Logistic Regression Balanced,0.5,0.741,0.340,0.64,0.444,0.770,0,8,26
7,Modelo 4 — Experiência Profissional,Logistic Regression Balanced,0.5,0.722,0.327,0.68,0.442,0.771,4,6,19
8,Modelo 8 — Integrado Multidimensional,Logistic Regression,0.5,0.861,0.630,0.34,0.442,0.834,7,11,43
9,Modelo 2 — Nível Hierárquico,Logistic Regression Balanced,0.5,0.709,0.318,0.70,0.438,0.768,0,8,22


In [61]:
top_combinations = (
    cv_comparison_sorted
    .head(20)[["Variable_Set", "Model"]]
    .to_dict("records")
)

top_combinations

[{'Variable_Set': 'Modelo 8 — Integrado Multidimensional',
  'Model': 'Logistic Regression Balanced'},
 {'Variable_Set': 'Modelo 8 — Integrado Multidimensional',
  'Model': 'Logistic Regression'},
 {'Variable_Set': 'Modelo 2 — Nível Hierárquico e Benefícios',
  'Model': 'Logistic Regression Balanced'},
 {'Variable_Set': 'Modelo 4 — Trajetória Organizacional',
  'Model': 'Logistic Regression Balanced'},
 {'Variable_Set': 'Modelo 5 — Antiguidade Organizacional',
  'Model': 'Logistic Regression Balanced'},
 {'Variable_Set': 'Modelo 6 — Perfil Pessoal',
  'Model': 'Logistic Regression Balanced'},
 {'Variable_Set': 'Modelo 4 — Experiência Profissional',
  'Model': 'Logistic Regression Balanced'},
 {'Variable_Set': 'Modelo 5 — Estabilidade e Benefícios',
  'Model': 'Logistic Regression Balanced'},
 {'Variable_Set': 'Modelo 6 — Perfil Pessoal e Condições de Trabalho',
  'Model': 'Logistic Regression Balanced'},
 {'Variable_Set': 'Modelo 1 — Função Profissional',
  'Model': 'Logistic Regressio

### Hyperparameter tuning

Hyperparameter tuning was performed to improve model performance and select the best configuration for each variable set and estimator.

Although the integrated multidimensional model achieved the highest tuning score, the hierarchical and benefits model remained one of the strongest candidates when considering interpretability, stability, and business relevance.

This reinforces the idea that the final model should not be selected solely based on the highest score, but through a balance between predictive performance and practical usefulness.

In [62]:
tuning_results_df, model_2 = tune_logistic_hyperparameters_top_combinations(
    df=df_train,
    models_dict=all_logistic_models_dict_c,
    estimators_dict=estimators_dict,
    param_distributions_dict=param_distributions_dict,
    top_combinations=top_combinations,
    target=target,
    n_iter=30,
    n_splits=10,
    scoring="f1",
    random_state=42,
    scale_numeric=True
)

tuning_results_sorted = (
    tuning_results_df
    .sort_values(
        by="Best_Score",
        ascending=False
    )
    .reset_index(drop=True)
)

tuning_results_sorted

/Users/nathansperinde/Desktop/Portifolio/projeto_1/.venv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
/Users/nathansperinde/Desktop/Portifolio/projeto_1/.venv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1429: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(
/Users/nathansperinde/Desktop/Portifolio/projeto_1/.venv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning

,Variable_Set,Model,Best_Score,Scoring,Best_Params,N_Parameter_Combinations_Tested,N_Numeric_Variables,N_Categorical_Variables,N_Features_After_Dummies
0,Modelo 8 — Integrado Multidimensional,Logistic Regression,0.542,f1,"{'classifier__solver': 'liblinear', 'classifie...",10,7,11,43
1,Modelo 8 — Integrado Multidimensional,Logistic Regression Balanced,0.531,f1,"{'classifier__solver': 'liblinear', 'classifie...",10,7,11,43
2,Modelo 2 — Nível Hierárquico e Benefícios,Logistic Regression Balanced,0.484,f1,"{'classifier__solver': 'liblinear', 'classifie...",10,3,8,25
3,Modelo 4 — Trajetória Organizacional,Logistic Regression Balanced,0.482,f1,"{'classifier__solver': 'liblinear', 'classifie...",10,0,8,22
4,Modelo 6 — Perfil Pessoal,Logistic Regression Balanced,0.470,f1,"{'classifier__solver': 'liblinear', 'classifie...",10,0,9,24
5,Modelo 4 — Experiência Profissional,Logistic Regression Balanced,0.467,f1,"{'classifier__solver': 'liblinear', 'classifie...",10,4,6,19
6,Modelo 5 — Antiguidade Organizacional,Logistic Regression Balanced,0.465,f1,"{'classifier__solver': 'liblinear', 'classifie...",10,5,6,20
7,Modelo 5 — Estabilidade e Benefícios,Logistic Regression Balanced,0.465,f1,"{'classifier__solver': 'liblinear', 'classifie...",10,0,9,24
8,Modelo 6 — Perfil Pessoal e Condições de Trabalho,Logistic Regression Balanced,0.465,f1,"{'classifier__solver': 'liblinear', 'classifie...",10,3,8,24
9,Modelo 1 — Função Profissional,Logistic Regression Balanced,0.460,f1,"{'classifier__solver': 'liblinear', 'classifie...",10,0,8,26


### Threshold optimization and final test evaluation

After tuning, thresholds were optimized to improve the balance between precision, recall, and F1-score.

The integrated multidimensional model achieved the highest F1-score on the test set, but it was treated as a control model due to its complexity. The hierarchical and benefits model with balanced logistic regression achieved strong test performance while remaining easier to interpret.

The final selected model achieved a good balance between accuracy, precision, recall, F1-score, AUC, and generalization stability.

In [63]:
(
    threshold_cv_comparison,
    best_thresholds_cv,
    final_test_results_df,
    confusion_results_opt,
    fitted_models,
    interpretation_results_opt
) = evaluate_thresholds_optimized_logistic_models_cv(
    df=df_train,
    df_test=df_test,
    models_dict=all_logistic_models_dict_c,
    best_models=model_2,
    target=target,
    thresholds=np.arange(0.20, 0.751, 0.01),
    test_size=0.30,
    random_state=42,
    n_splits=10,
    threshold_metric="F1-score"
)

best_thresholds_cv

/Users/nathansperinde/Desktop/Portifolio/projeto_1/.venv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
/Users/nathansperinde/Desktop/Portifolio/projeto_1/.venv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penal

,Variable_Set,Model,Threshold,Accuracy,Precision,Recall,F1-score,AUC
1,Modelo 8 — Integrado Multidimensional,Logistic Regression,0.38,0.874,0.615,0.578,0.596,0.853
0,Modelo 8 — Integrado Multidimensional,Logistic Regression Balanced,0.64,0.846,0.518,0.681,0.589,0.842
16,Modelo 2 — Nível Hierárquico e Benefícios,Logistic Regression,0.33,0.848,0.530,0.524,0.527,0.812
2,Modelo 2 — Nível Hierárquico e Benefícios,Logistic Regression Balanced,0.71,0.848,0.532,0.506,0.519,0.807
8,Modelo 6 — Perfil Pessoal e Condições de Trabalho,Logistic Regression Balanced,0.59,0.801,0.424,0.651,0.513,0.790
17,Modelo 5 — Estabilidade e Benefícios,Logistic Regression,0.23,0.807,0.430,0.608,0.504,0.798
7,Modelo 5 — Estabilidade e Benefícios,Logistic Regression Balanced,0.74,0.856,0.568,0.452,0.503,0.791
3,Modelo 4 — Trajetória Organizacional,Logistic Regression Balanced,0.57,0.791,0.408,0.651,0.501,0.792
18,Modelo 4 — Trajetória Organizacional,Logistic Regression,0.26,0.824,0.462,0.542,0.499,0.795
5,Modelo 6 — Perfil Pessoal,Logistic Regression Balanced,0.56,0.780,0.394,0.675,0.498,0.786


### Final model selection

The final evaluation suggests two possible model selection strategies, depending on the business objective.

The first option is the **main balanced and interpretable model**:

**Model:** Modelo 2 — Nível Hierárquico e Benefícios  
**Estimator:** Logistic Regression Balanced  
**Threshold:** 0.71  

This model was selected as the main final model because it provides the best balance between predictive performance, interpretability, and generalization stability. It achieved strong accuracy, precision, F1-score, and AUC, while keeping the model structure relatively simple and aligned with the exploratory analysis.

The second option is a **recall-oriented model**, focused on reducing false negatives:

**Model:** Modelo 3 — Faixa Salarial  
**Estimator:** Logistic Regression Balanced  
**Threshold:** 0.58  

This model is useful when the business priority is to identify more employees at risk of leaving, even if this increases the number of false positives. It achieved a higher recall than the main selected model and reduced the number of false negatives, while still maintaining acceptable F1-score and AUC.

The integrated multidimensional model was not selected as a final model. It was kept as a control model because it combines a larger number of predictors and serves as a benchmark for maximum predictive performance. However, due to its higher complexity and greater risk of multicollinearity, it is less suitable for final business interpretation.

In [64]:
final_test_results_sorted = (
    final_test_results_df
    .sort_values(
        by=["F1-score", "Recall", "AUC"],
        ascending=False
    )
    .reset_index(drop=True)
)

final_test_results_sorted

,Variable_Set,Model,Threshold,Accuracy,Precision,Recall,F1-score,AUC
0,Modelo 8 — Integrado Multidimensional,Logistic Regression,0.38,0.864,0.575,0.592,0.583,0.814
1,Modelo 2 — Nível Hierárquico e Benefícios,Logistic Regression Balanced,0.71,0.864,0.582,0.549,0.565,0.830
2,Modelo 8 — Integrado Multidimensional,Logistic Regression Balanced,0.64,0.837,0.495,0.634,0.556,0.810
3,Modelo 2 — Nível Hierárquico e Benefícios,Logistic Regression,0.33,0.853,0.543,0.535,0.539,0.833
4,Modelo 3 — Faixa Salarial,Logistic Regression Balanced,0.58,0.810,0.437,0.634,0.517,0.797
5,Modelo 2 — Nível Hierárquico,Logistic Regression Balanced,0.62,0.823,0.461,0.577,0.512,0.817
6,Modelo 6 — Perfil Pessoal e Condições de Trabalho,Logistic Regression Balanced,0.59,0.816,0.447,0.592,0.509,0.802
7,Modelo 4 — Experiência Profissional,Logistic Regression Balanced,0.61,0.821,0.456,0.577,0.509,0.796
8,Modelo 5 — Antiguidade Organizacional,Logistic Regression Balanced,0.60,0.807,0.430,0.606,0.503,0.781
9,Modelo 6 — Perfil Pessoal,Logistic Regression Balanced,0.56,0.794,0.407,0.620,0.492,0.802


In [65]:
final_summary = (
    final_test_results_sorted
    .merge(
        tuning_results_sorted[
            ["Variable_Set", "Model", "Best_Score", "Best_Params"]
        ],
        on=["Variable_Set", "Model"],
        how="left"
    )
    .merge(
        gap_analysis_sorted[
            [
                "Variable_Set",
                "Model",
                "Train_F1_Mean",
                "CV_F1_Mean",
                "F1_Gap",
                "Train_Recall_Mean",
                "CV_Recall_Mean",
                "Recall_Gap",
                "Gap_Diagnosis"
            ]
        ],
        on=["Variable_Set", "Model"],
        how="left"
    )
)

final_summary

,Variable_Set,Model,Threshold,Accuracy,Precision,Recall,F1-score,AUC,Best_Score,Best_Params,Train_F1_Mean,CV_F1_Mean,F1_Gap,Train_Recall_Mean,CV_Recall_Mean,Recall_Gap,Gap_Diagnosis
0,Modelo 8 — Integrado Multidimensional,Logistic Regression,0.38,0.864,0.575,0.592,0.583,0.814,0.542,"{'classifier__solver': 'liblinear', 'classifie...",0.606,0.526,0.080,0.483,0.422,0.062,Moderate gap
1,Modelo 2 — Nível Hierárquico e Benefícios,Logistic Regression Balanced,0.71,0.864,0.582,0.549,0.565,0.830,0.484,"{'classifier__solver': 'liblinear', 'classifie...",0.515,0.478,0.037,0.780,0.716,0.064,Stable generalization
2,Modelo 8 — Integrado Multidimensional,Logistic Regression Balanced,0.64,0.837,0.495,0.634,0.556,0.810,0.531,"{'classifier__solver': 'liblinear', 'classifie...",0.571,0.531,0.041,0.823,0.758,0.065,Stable generalization
3,Modelo 2 — Nível Hierárquico e Benefícios,Logistic Regression,0.33,0.853,0.543,0.535,0.539,0.833,0.405,"{'classifier__solver': 'liblinear', 'classifie...",0.420,0.388,0.032,0.292,0.271,0.021,Stable generalization
4,Modelo 3 — Faixa Salarial,Logistic Regression Balanced,0.58,0.810,0.437,0.634,0.517,0.797,0.442,"{'classifier__solver': 'liblinear', 'classifie...",0.475,0.440,0.035,0.731,0.679,0.052,Stable generalization
5,Modelo 2 — Nível Hierárquico,Logistic Regression Balanced,0.62,0.823,0.461,0.577,0.512,0.817,0.451,"{'classifier__solver': 'liblinear', 'classifie...",0.468,0.448,0.020,0.722,0.692,0.031,Stable generalization
6,Modelo 6 — Perfil Pessoal e Condições de Trabalho,Logistic Regression Balanced,0.59,0.816,0.447,0.592,0.509,0.802,0.465,"{'classifier__solver': 'liblinear', 'classifie...",0.500,0.458,0.041,0.756,0.703,0.053,Stable generalization
7,Modelo 4 — Experiência Profissional,Logistic Regression Balanced,0.61,0.821,0.456,0.577,0.509,0.796,0.467,"{'classifier__solver': 'liblinear', 'classifie...",0.492,0.459,0.034,0.765,0.721,0.044,Stable generalization
8,Modelo 5 — Antiguidade Organizacional,Logistic Regression Balanced,0.60,0.807,0.430,0.606,0.503,0.781,0.465,"{'classifier__solver': 'liblinear', 'classifie...",0.480,0.464,0.016,0.760,0.733,0.027,Stable generalization
9,Modelo 6 — Perfil Pessoal,Logistic Regression Balanced,0.56,0.794,0.407,0.620,0.492,0.802,0.470,"{'classifier__solver': 'liblinear', 'classifie...",0.504,0.464,0.040,0.757,0.704,0.053,Stable generalization


### Final model performance

Two model options were considered in the final evaluation.

The main balanced and interpretable model was:

**Modelo 2 — Nível Hierárquico e Benefícios | Logistic Regression Balanced | Threshold = 0.71**

Its test performance was:

- Accuracy: 0.864
- Precision: 0.582
- Recall: 0.549
- F1-score: 0.565
- AUC: 0.830

This model achieved the strongest balance between performance and interpretability. It is suitable when the goal is to support decision-making with a model that is stable, explainable, and reasonably balanced across different metrics.

The alternative recall-oriented model was:

**Modelo 3 — Faixa Salarial | Logistic Regression Balanced | Threshold = 0.58**

Its test performance was:

- Accuracy: 0.810
- Precision: 0.437
- Recall: 0.634
- F1-score: 0.517
- AUC: 0.797

This model achieved higher recall, meaning that it identified a larger proportion of employees who actually left the company. However, this came with lower precision and accuracy, meaning that more employees were incorrectly flagged as at risk.

Therefore, the final choice depends on the business objective. If the priority is overall balance and interpretability, the hierarchical and benefits model is preferable. If the priority is reducing false negatives and detecting more at-risk employees, the salary range model becomes a relevant operational alternative.

In [66]:
model_2 = final_summary.iloc[1]

print("Variable set:", model_2["Variable_Set"])
print("Model:", model_2["Model"])
print("Best threshold:", model_2["Threshold"])

confusion_mat_m2 = confusion_results_opt[(model_2["Variable_Set"], model_2["Model"])]

interpretation_m2 = interpretation_results_opt[(model_2["Variable_Set"], model_2["Model"])]

top_positive_odds_m2 = (interpretation_m2.sort_values("Odds_Ratio", ascending=False).head(20).reset_index(drop=True))

top_negative_odds_m2 = (interpretation_m2.sort_values("Odds_Ratio", ascending=True).head(20).reset_index(drop=True))

Variable set: Modelo 2 — Nível Hierárquico e Benefícios
Model: Logistic Regression Balanced
Best threshold: 0.71


In [67]:
model_3 = final_summary.iloc[4]

print("Variable set:", model_3["Variable_Set"])
print("Model:", model_3["Model"])
print("Best threshold:", model_3["Threshold"])

confusion_mat_m3 = confusion_results_opt[(model_3["Variable_Set"], model_3["Model"])]

interpretation_m3 = interpretation_results_opt[(model_3["Variable_Set"], model_3["Model"])]

top_positive_odds_m3 = (interpretation_m3.sort_values("Odds_Ratio", ascending=False).head(20).reset_index(drop=True))

top_negative_odds_m3 = (interpretation_m3.sort_values("Odds_Ratio", ascending=True).head(20).reset_index(drop=True))

Variable set: Modelo 3 — Faixa Salarial
Model: Logistic Regression Balanced
Best threshold: 0.58


In [68]:
display(model_2)

print("\n\n")

display(model_3)


Variable_Set                 Modelo 2 — Nível Hierárquico e Benefícios
Model                                     Logistic Regression Balanced
Threshold                                                         0.71
Accuracy                                                         0.864
Precision                                                        0.582
Recall                                                           0.549
F1-score                                                         0.565
AUC                                                               0.83
Best_Score                                                       0.484
Best_Params          {'classifier__solver': 'liblinear', 'classifie...
Train_F1_Mean                                                    0.515
CV_F1_Mean                                                       0.478
F1_Gap                                                           0.037
Train_Recall_Mean                                                 0.78
CV_Rec

Variable_Set                                 Modelo 3 — Faixa Salarial
Model                                     Logistic Regression Balanced
Threshold                                                         0.58
Accuracy                                                          0.81
Precision                                                        0.437
Recall                                                           0.634
F1-score                                                         0.517
AUC                                                              0.797
Best_Score                                                       0.442
Best_Params          {'classifier__solver': 'liblinear', 'classifie...
Train_F1_Mean                                                    0.475
CV_F1_Mean                                                        0.44
F1_Gap                                                           0.035
Train_Recall_Mean                                                0.731
CV_Rec

### Final odds ratio interpretation

The odds ratios of the main selected model show that the strongest factors associated with higher attrition risk are frequent business travel, overtime, low job involvement, low environment satisfaction, no stock options, low job satisfaction, and greater distance from home.

For the main balanced and interpretable model, the strongest risk factors were:

- `BusinessTravel_Travel_Frequently`
- `OverTime_Yes`
- `JobInvolvementLevel_Low`
- `EnvironmentSatisfactionLevel_Low`
- `StockOption_No Options`
- `SatisfactionLevel_Low`
- `DistanceFromHome`

The strongest protective factors were related to higher job levels, better work-life balance, higher satisfaction, stronger job involvement, and stock option availability.

The recall-oriented salary range model shows a similar pattern, but also places stronger emphasis on income-related risk. In this model, `IncomeGroup_Low` appears as one of the strongest predictors of higher attrition risk, while `IncomeGroup_Very High` appears as a protective factor.

This reinforces the idea that attrition is associated with a combination of work demands, employee engagement, satisfaction, benefits, job level, income, and commuting distance. The consistency of these patterns across models strengthens the interpretation of the results.

In [69]:
display(interpretation_m2)

print("\n\n")

display(interpretation_m3)

,Feature,Coefficient,Odds_Ratio
0,BusinessTravel_Travel_Frequently,1.520391,4.574015
1,OverTime_Yes,1.448792,4.257968
2,JobInvolvementLevel_Low,1.406987,4.083634
3,BusinessTravel_Travel_Rarely,0.837882,2.311466
4,EnvironmentSatisfactionLevel_Low,0.785344,2.193162
5,StockOption_No Options,0.550660,1.734398
6,SatisfactionLevel_Low,0.414021,1.512889
7,SatisfactionLevel_Medium,0.304553,1.356019
8,DistanceFromHome,0.281114,1.324605
9,JobInvolvementLevel_Medium,0.061747,1.063693


,Feature,Coefficient,Odds_Ratio
0,JobInvolvementLevel_Low,1.487769,4.427208
1,OverTime_Yes,1.425061,4.158110
2,BusinessTravel_Travel_Frequently,1.354520,3.874900
3,IncomeGroup_Low,1.076484,2.934345
4,EnvironmentSatisfactionLevel_Low,0.921126,2.512117
5,BusinessTravel_Travel_Rarely,0.808493,2.244523
6,DistanceGroup_21-30,0.552475,1.737549
7,SatisfactionLevel_Low,0.492775,1.636853
8,DistanceGroup_11-20,0.379846,1.462059
9,SatisfactionLevel_Medium,0.210305,1.234054


### Final confusion matrix interpretation

The confusion matrices highlight the trade-off between the two final model strategies.

For the main balanced and interpretable model:

**Modelo 2 — Nível Hierárquico e Benefícios | Logistic Regression Balanced | Threshold = 0.71**

The confusion matrix was:

- True negatives: 342
- False positives: 28
- False negatives: 32
- True positives: 39

This model correctly classified most employees who did not leave and maintained a relatively low number of false positives. However, it missed 32 employees who actually left, which represents its main limitation in a retention context.

For the recall-oriented alternative:

**Modelo 3 — Faixa Salarial | Logistic Regression Balanced | Threshold = 0.58**

The confusion matrix was:

- True negatives: 312
- False positives: 58
- False negatives: 26
- True positives: 45

This model reduced the number of false negatives from 32 to 26 and identified more employees who actually left. However, it also increased the number of false positives from 28 to 58.

In an attrition context, false negatives are particularly important because they represent employees who were predicted to stay but actually left. If the organization wants to minimize missed attrition cases, the recall-oriented model may be more appropriate. If the organization wants a more balanced and precise model, the hierarchical and benefits model is preferable.

In [70]:
display(confusion_mat_m2)

print("\n\n")

display(confusion_mat_m3)

array([[342,  28],
       [ 32,  39]])

array([[312,  58],
       [ 26,  45]])

### Business interpretation

The final results support two possible decision-making strategies.

The first strategy is to use the **hierarchical and benefits model** as the main decision-support model. This option is more balanced and interpretable. It is useful when the organization wants a model that performs well across accuracy, precision, recall, F1-score, and AUC, while also producing clear business insights.

Under this strategy, the model suggests that employees with frequent business travel, overtime, low job involvement, low environment satisfaction, no stock options, low job satisfaction, and greater distance from home are more likely to be at risk of attrition. These findings can support retention actions related to workload, engagement, satisfaction, benefits, and employee support.

The second strategy is to use the **salary range model** when the main objective is to reduce false negatives. This option is more recall-oriented and identifies more employees who actually leave. It may be useful when the organization prefers to flag more employees as potentially at risk in order to avoid missing relevant attrition cases.

However, this approach also increases false positives. In practice, this means that more employees who would not leave may still be flagged as at risk. This may be acceptable if the intervention is low-cost, such as additional check-ins, satisfaction surveys, or manager follow-ups.

Therefore, the model should be used as a decision-support tool rather than an automatic decision system. The final strategy should depend on the organization's tolerance for false positives and false negatives.

### Limitations

This analysis has some limitations. First, the dataset is imbalanced, which makes the identification of attrition cases more challenging. Second, some predictors are strongly related to each other, which can affect coefficient stability and interpretation.

Additionally, the dataset is observational, so the results should be interpreted as associations rather than causal effects. The model can identify patterns related to attrition, but it cannot prove that a specific factor directly causes employees to leave.

Finally, the model should be used as a decision-support tool. Any practical application should involve human evaluation and ethical consideration, especially because employee-level predictions can affect people directly.

### Final conclusion

The final results show that logistic regression can provide both predictive performance and interpretability for employee attrition analysis.

The integrated multidimensional model achieved strong predictive performance, but it was treated only as a control model. Its purpose was to provide a benchmark for the maximum performance obtained when many predictors are combined. Due to its higher complexity and greater risk of multicollinearity, it was not selected as the final interpretive model.

Two practical model choices were identified.

The first is the **main balanced and interpretable model**: `Modelo 2 — Nível Hierárquico e Benefícios` with balanced logistic regression and a threshold of 0.71. This model offers the best balance between predictive performance, stability, interpretability, and business relevance.

The second is the **recall-oriented alternative**: `Modelo 3 — Faixa Salarial` with balanced logistic regression and a threshold of 0.58. This model is useful when the main priority is to reduce false negatives and identify more employees who are likely to leave, even at the cost of increasing false positives.

Overall, the analysis suggests that attrition is associated with a combination of overtime, frequent business travel, low job involvement, low satisfaction, lack of stock options, lower income, lower job level, poorer work-life balance, and greater distance from home.

The final decision between the two model strategies should depend on the business objective. If the priority is balanced and explainable decision support, the hierarchical and benefits model is preferable. If the priority is early detection and reducing missed attrition cases, the salary range model becomes a stronger operational alternative.

In [71]:
from pathlib import Path

TABLES_DIR = Path("../../outputs/tables")

In [72]:
cv_comparison_sorted.to_csv(
    TABLES_DIR / "final_cross_validation_results.csv", 
    index=False
    )

gap_analysis_sorted.to_csv(
    TABLES_DIR / "final_gap_analysis.csv", 
    index=False
    )

general_results_sorted.to_csv(
    TABLES_DIR / "final_general_results.csv", 
    index=False
    )

tuning_results_sorted.to_csv(
    TABLES_DIR / "final_tuning_results.csv", 
    index=False
    )

best_thresholds_cv.to_csv(
    TABLES_DIR / "final_best_thresholds_cv.csv", 
    index=False
    )

final_test_results_sorted.to_csv(
    TABLES_DIR / "final_test_results.csv", 
    index=False
    )

final_summary.to_csv(
    TABLES_DIR / "final_summary.csv", 
    index=False
    )

interpretation_m2.to_csv(
    TABLES_DIR / "model_2_odds_ratios.csv",
    index=False
)

interpretation_m3.to_csv(
    TABLES_DIR / "model_3_odds_ratios.csv",
    index=False
)